# Datheon Python + FastAPI Backend Internship
## Task 3: Database Persistence with SQLAlchemy & SQLite
**Application ID:** `DT-2026-89H4F` | **Candidate:** Syed Jawad Hussain Shah


In [1]:
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

SQLALCHEMY_DATABASE_URL = "sqlite:///./datheon_persistence.db"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False}
)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


from datetime import datetime, timezone
from sqlalchemy import Column, Integer, String, Boolean, DateTime, ForeignKey, Text
from sqlalchemy.orm import relationship
from database import Base

class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String(50), unique=True, index=True, nullable=False)
    email = Column(String(100), unique=True, index=True, nullable=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))

    notes = relationship("Note", back_populates="owner", cascade="all, delete-orphan")

class Note(Base):
    __tablename__ = "notes"

    id = Column(Integer, primary_key=True, index=True)
    title = Column(String(150), index=True, nullable=False)
    content = Column(Text, nullable=False)
    is_archived = Column(Boolean, default=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))
    user_id = Column(Integer, ForeignKey("users.id"), nullable=False)

    owner = relationship("User", back_populates="notes")


In [1]:
"""
Datheon Python + FastAPI Backend Development Internship
Task 3: Database Persistence with SQLAlchemy & SQLite
Application ID: DT-2026-89H4F | Candidate: Syed Jawad Hussain Shah
"""

from typing import List
from fastapi import FastAPI, Depends, HTTPException, status
from sqlalchemy.orm import Session
import models, schemas, crud
from database import engine, get_db

models.Base.metadata.create_all(bind=engine)

app = FastAPI(
    title="Datheon SQLAlchemy Relational Persistence API",
    description="Relational database persistence layer with User -> Notes relationships and Foreign Key joins.",
    version="1.0.0"
)

@app.post("/users", response_model=schemas.UserResponse, status_code=status.HTTP_201_CREATED, tags=["Users"])
def create_user(user: schemas.UserCreate, db: Session = Depends(get_db)):
    db_user = crud.get_user_by_username(db, username=user.username)
    if db_user:
        raise HTTPException(status_code=400, detail="Username already exists")
    return crud.create_user(db=db, user=user)

@app.get("/users", response_model=List[schemas.UserResponse], tags=["Users"])
def read_users(skip: int = 0, limit: int = 100, db: Session = Depends(get_db)):
    return crud.get_users(db, skip=skip, limit=limit)

@app.get("/users/{user_id}", response_model=schemas.UserDetailResponse, tags=["Users"])
def read_user(user_id: int, db: Session = Depends(get_db)):
    db_user = crud.get_user(db, user_id=user_id)
    if db_user is None:
        raise HTTPException(status_code=404, detail="User not found")
    return db_user

@app.post("/users/{user_id}/notes", response_model=schemas.NoteResponse, status_code=status.HTTP_201_CREATED, tags=["Notes"])
def create_note_for_user(user_id: int, note: schemas.NoteCreate, db: Session = Depends(get_db)):
    db_user = crud.get_user(db, user_id=user_id)
    if not db_user:
        raise HTTPException(status_code=404, detail="User not found")
    return crud.create_user_note(db=db, note=note, user_id=user_id)

@app.get("/notes", response_model=List[schemas.NoteResponse], tags=["Notes"])
def read_notes(skip: int = 0, limit: int = 100, db: Session = Depends(get_db)):
    return crud.get_notes(db, skip=skip, limit=limit)


In [1]:
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def test_sql_persistence():
    u_resp = client.post("/users", json={"username": "jawad_ai", "email": "jawad@biit.edu.pk"})
    assert u_resp.status_code in (201, 400)
    user_id = u_resp.json()["id"] if u_resp.status_code == 201 else 1

    note_resp = client.post(f"/users/{user_id}/notes", json={
        "title": "FastAPI + SQLAlchemy Architecture",
        "content": "Implemented clean declarative relational models with foreign keys.",
        "is_archived": False
    })
    assert note_resp.status_code == 201
    assert note_resp.json()["user_id"] == user_id

    user_detail = client.get(f"/users/{user_id}")
    assert user_detail.status_code == 200
    assert len(user_detail.json()["notes"]) >= 1

if __name__ == "__main__":
    test_sql_persistence()
    print("ALL TASK 3 SQLALCHEMY TESTS PASSED SUCCESSFULLY!")


ALL TASK 3 SQLALCHEMY TESTS PASSED SUCCESSFULLY!
